In [5]:
import os

# Set this to  actual project path
project_path = r"c:\Users\saqib\OneDrive\Desktop\production-grade-wine-predictor"

os.chdir(project_path)

print(f"Now at: {os.getcwd()}")
print(f"Does 'config' exist here? {os.path.exists('config')}")

Now at: c:\Users\saqib\OneDrive\Desktop\production-grade-wine-predictor
Does 'config' exist here? True


In [6]:
%pwd

'c:\\Users\\saqib\\OneDrive\\Desktop\\production-grade-wine-predictor'

In [8]:
!pip install pandas


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 2.7 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/9.9 MB 3.7 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.9 MB 3.1 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.9 MB 2.7 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.9 MB 2.7 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.9 MB 2.7 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.9 MB 2.7 MB/s eta 0:00:03
   --------- ------------------------------ 2.4/9.9 MB 1.4 MB/s eta 0:00:06
   ----------- ---------------------------- 2.9/9.9 MB 1.5 MB/s eta 0:00:05
   ----------- ---------------------------- 2.9/9.9 MB 1.5 MB/s eta 0:00:05
   -------------- ------------------------- 3.7/9.9 MB 1.7 MB/s eta 0:00:04
   -------------- --------

In [9]:
import pandas as pd

data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")

In [10]:
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [ ]:
data.info()
# then update the schema.yaml file with the correct number of columns and their names.

<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [13]:
data.isnull().sum()
# if there are any null values, we can handle them here and then update the data_validation notebook with the steps to handle null values.

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [15]:
data.shape

(1599, 12)

In [16]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir:Path
    STATUS_FILE:str
    unzip_data_dir:Path
    all_schema:dict

In [17]:
from src.productiongradewinepredictor.constants import *
from src.productiongradewinepredictor.utils.common import read_yaml, create_directories

In [18]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config

In [19]:
import os
from src.productiongradewinepredictor import logger

In [20]:
class DataValiadtion:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self)-> bool:
        try:
            validation_status = None

            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()

            
            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

            return validation_status
        
        except Exception as e:
            raise e


In [21]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValiadtion(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2026-03-25 00:20:23,756: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-25 00:20:23,763: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-25 00:20:23,766: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-25 00:20:23,769: INFO: common: created directory at: artifacts]
[2026-03-25 00:20:23,771: INFO: common: created directory at: artifacts/data_validation]
